# Observability and Tracing

Observability helps you understand what's happening inside your agent workflows. This tutorial covers how to add tracing and monitoring to your NAT workflows.

## What You'll Learn

1. Understanding observability in AI agents
2. Configuring Phoenix/Arize tracing
3. Viewing traces in the Phoenix UI
4. Analyzing agent behavior
5. Debugging workflow issues

## Why Observability?

- **Debug issues** - Understand why an agent made a decision
- **Monitor performance** - Track latency and token usage
- **Audit decisions** - Record agent reasoning for compliance
- **Optimize costs** - Identify expensive operations


In [ ]:
import sys
from pathlib import Path

# Setup
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


## Prerequisites

Before using observability features, install Phoenix:

```bash
pip install arize-phoenix
```

Start the Phoenix server:

```bash
phoenix serve
```

This will start the Phoenix UI at `http://localhost:6006`


## Step 1: Create a Workflow


In [ ]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create tools
time_tool = CurrentTimeTool(name="current_time")

try:
    from nat_simple_calculator.register import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
except ImportError:
    tools = [time_tool]

# Create agent
agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
)

workflow = NatWorkflow(entrypoint=agent)
print("✅ Workflow created")


## Step 2: Configure Tracing

NAT supports Phoenix/Arize for distributed tracing:


In [ ]:
# Phoenix tracing is enabled via environment variable or config
# Option 1: Environment variable
# os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006/v1/traces"

# Option 2: The workflow will automatically detect Phoenix if available
# and send traces when you run the workflow

print("ℹ️  To enable tracing:")
print("   1. Start Phoenix: phoenix serve")
print("   2. Run your workflow - traces will be sent automatically")
print("   3. View traces at: http://localhost:6006")


## Step 3: Save Configuration


In [ ]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "traced_workflow.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")


## Step 4: Run with Tracing

### Via Python


In [ ]:
# Run workflow (uncomment to execute)
# Traces will be sent to Phoenix automatically if it's running
# result = await workflow.prompt("What is 25 * 4?")
# print(result)


### Via CLI

```bash
# Make sure Phoenix is running first
phoenix serve

# Then run your workflow
nat run --config_file configs/traced_workflow.yaml \
    --input "What is 25 * 4?"

# View traces at http://localhost:6006
```


## Understanding Traces

In the Phoenix UI, you'll see:

### Trace Structure
```
📊 Workflow Execution
├── 🤖 Agent: ReActAgent
│   ├── 💭 LLM Call (reasoning)
│   │   ├── Input tokens: 150
│   │   ├── Output tokens: 50
│   │   └── Latency: 1.2s
│   ├── 🔧 Tool Call: calculator.multiply
│   │   ├── Input: {"a": 25, "b": 4}
│   │   ├── Output: 100
│   │   └── Latency: 5ms
│   └── 💭 LLM Call (final response)
│       ├── Input tokens: 200
│       ├── Output tokens: 30
│       └── Latency: 0.8s
└── ✅ Result: "25 * 4 = 100"
```

### Key Metrics
- **Latency** - Time taken for each operation
- **Token Usage** - Input/output tokens per LLM call
- **Tool Calls** - Which tools were invoked and their results
- **Error Traces** - Failed operations and error messages


## Debugging with Traces

Traces help you debug common issues:

### Issue: Agent Not Using Tools
Look at the LLM reasoning spans to see if:
- Tools are mentioned in the prompt
- Tool descriptions are clear
- Agent understood the task

### Issue: Slow Performance
Check latency breakdown:
- LLM calls are usually the slowest
- Tool calls should be fast
- Network issues show high latency

### Issue: Wrong Answers
Examine the reasoning chain:
- What tools were called?
- What were the tool results?
- How did the LLM interpret results?


## Summary

In this tutorial, you learned:

✅ Why observability matters for AI agents  
✅ Setting up Phoenix tracing  
✅ Running workflows with tracing enabled  
✅ Understanding trace structure  
✅ Debugging with trace data  

## Next Steps

- **[09_evaluation.ipynb](./09_evaluation.ipynb)** - Evaluate workflow performance
- **[08_configuration_guide.ipynb](./08_configuration_guide.ipynb)** - Deep dive into configuration

## Additional Resources

- [Phoenix Documentation](https://docs.arize.com/phoenix)
- [OpenTelemetry Tracing](https://opentelemetry.io/docs/)
